# Inference Production Runner

Run `inference_production` using the trained checkpoint and input parquet files in `data_v2`.

In [1]:
from pathlib import Path
import torch

# --- Paths (WSL) ---
repo_root = Path('.').resolve()
model_dir_wsl = Path('/home/sagemaker-user/congestion-bogota/artifacts/model_tgcn_2026_01_27')
data_v2_wsl = Path('/home/sagemaker-user/congestion-bogota/data_v2')

if model_dir_wsl.exists() and data_v2_wsl.exists():
    model_dir = model_dir_wsl
    data_v2 = data_v2_wsl
    base_repo = Path('/home/sagemaker-user/congestion-bogota')
else:
    model_dir = repo_root / 'data' / 'artifacts' / 'model_tgcn_2026_01_27'
    data_v2 = repo_root / 'data_v2'
    base_repo = repo_root

model_path = model_dir / 'model.pth'

# Data inputs (parquets)
jams_path = data_v2 / 'df_jams_test.parquet'
irregularities_path = data_v2 / 'df_irregularidades_test.parquet'
weather_path = data_v2 / 'df_clima_test.parquet'
pmt_puntos_path = data_v2 / 'df_pmt_puntos_test.parquet'
pmt_tramos_path = data_v2 / 'df_pmt_tramos_test.parquet'
events_path = data_v2 / 'df_eventos_test.parquet'
incidents_path = data_v2 / "df_incidentes_test.parquet"

# Shared assets (graph/static/norm) from training preprocess
processed_data_dir = base_repo / 'data' / 'datos_procesados'
graph_path = processed_data_dir / 'conectivity_graph' / 'road_graph.pickle'
static_path = processed_data_dir / 'sparse_datasets' / 'static_dataset.h5'
norm_path = processed_data_dir / 'sparse_datasets' / 'normalization_constants.pkl'

# Optional output path
output_path = base_repo / 'data' / 'artifacts' / 'inference_preds.npy'

for p in [model_path, jams_path, irregularities_path, weather_path, pmt_puntos_path, pmt_tramos_path, events_path, graph_path, static_path, norm_path]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

print('Paths OK')


Paths OK


In [2]:
model_path

PosixPath('/home/sagemaker-user/congestion-bogota/artifacts/model_tgcn_2026_01_27/model.pth')

In [3]:
import sys
import pickle
import torch
from pathlib import Path

cwd = Path.cwd().resolve()
if cwd.name == 'notebooks':
    repo_root = cwd.parent
else:
    repo_root = cwd

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.models.gnn_temporal import GNN_Temporal_Model

# --- Load checkpoint ---
checkpoint = torch.load(model_path, weights_only=False)
model_kwargs = checkpoint.get('model_kwargs', {})
keys_to_drop = {"include_weather", "num_weather_features"}
for k in keys_to_drop:
    model_kwargs.pop(k, None)
state_dict = checkpoint.get('state_dict', None)

if state_dict is None:
    raise ValueError('Checkpoint missing state_dict. Cannot load model weights.')

# Build model
model = GNN_Temporal_Model(
    edge_index=None,
    **model_kwargs,
)
model.load_state_dict(state_dict)

# Move to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

print('Model loaded:', model.__class__.__name__, 'device=', device)
print('model_kwargs:', model_kwargs)


Model loaded: GNN_Temporal_Model device= cuda
model_kwargs: {'static_dim': 117, 'dynamic_dim': 124, 'future_dim': 5, 'input_seq_len': 150, 'forecast_horizon': 50, 'model_type': 'TGCN', 'include_velocity_target': True}


In [4]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if cwd.name == 'notebooks':
    repo_root = cwd.parent
else:
    repo_root = cwd

src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from src.inference_production import InferencePipeline

# Initialize pipeline without auto-loading model
pipeline = InferencePipeline(
    graph_path=str(graph_path),
    static_data_path=str(static_path),
    normalization_constants_path=str(norm_path),
    model_path=None,
    use_ray=False,
    expected_feature_map_path=  str('/home/sagemaker-user/congestion-bogota/artifacts/feature_maps/expected_feature_map.by_dataset.json'),
)

# Inject the model we loaded from checkpoint
pipeline.model = model

results = pipeline.predict(
    jams_path=str(jams_path),
    irregularities_path=str(irregularities_path),
    weather_path=str(weather_path),
    pmt_puntos_path=str(pmt_puntos_path),
    pmt_tramos_path=str(pmt_tramos_path),
    events_path=str(events_path),
    incidents_path=str(incidents_path),
    output_path=str(output_path),
)

print('Results keys:', results.keys())
print('num_batches:', results['num_batches'])
print('num_nodes:', results['num_nodes'])
print('load_time:', results['load_time'])
print('inference_time:', results['inference_time'])
print('total_time:', results['total_time'])

if results['predictions'] is not None:
    pred = results['predictions'][0]
    print('First prediction shape:', pred.shape)


2026-01-28 18:16:04,888 - src.inference_production - INFO - Initializing inference pipeline...
2026-01-28 18:16:19,068 - src.inference_production - INFO - Loading normalization constants from /home/sagemaker-user/congestion-bogota/data/datos_procesados/sparse_datasets/normalization_constants.pkl
2026-01-28 18:16:19,069 - src.inference_production - INFO - Loaded target normalization constants: ['mean_jam_level', 'mean_jam_velocity']
2026-01-28 18:16:19,069 - src.inference_production - INFO - Pipeline initialized with 139107 nodes
2026-01-28 18:16:19,070 - src.inference_production - INFO - Starting inference...
2026-01-28 18:16:19,070 - src.inference_production - INFO - Loading data...
2026-01-28 18:16:19,263 - inference_dataloader - INFO - Loaded jams: rows=610696 cols=12 in 0.19s
2026-01-28 18:16:19,270 - inference_dataloader - WARNING - Missing columns filled with 0: ['ACCIDENT_MINOR_count', 'ACCIDENT_MINOR_nThumbsUp', 'HAZARD_ON_ROAD_ICE_count', 'HAZARD_ON_ROAD_ICE_nThumbsUp', 'HAZAR

Results keys: dict_keys(['timestamp', 'num_batches', 'num_nodes', 'load_time', 'inference_time', 'total_time', 'predictions_normalized', 'predictions', 'normalization_constants'])
num_batches: 1
num_nodes: 500
load_time: 79.87255692481995
inference_time: 0.6008517742156982
total_time: 80.47644877433777
First prediction shape: torch.Size([1, 500, 50, 2])


In [5]:
print(model)

GNN_Temporal_Model(
  (static_encoder): StaticEncoder(
    (net): Sequential(
      (0): Linear(in_features=117, out_features=64, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=64, out_features=32, bias=True)
    )
  )
  (recurrent): TGCN(
    (conv_z): GCNConv(156, 32)
    (linear_z): Linear(in_features=64, out_features=32, bias=True)
    (conv_r): GCNConv(156, 32)
    (linear_r): Linear(in_features=64, out_features=32, bias=True)
    (conv_h): GCNConv(156, 32)
    (linear_h): Linear(in_features=64, out_features=32, bias=True)
  )
  (future_recurrent): TGCN(
    (conv_z): GCNConv(37, 32)
    (linear_z): Linear(in_features=64, out_features=32, bias=True)
    (conv_r): GCNConv(37, 32)
    (linear_r): Linear(in_features=64, out_features=32, bias=True)
    (conv_h): GCNConv(37, 32)
    (linear_h): Linear(in_features=64, out_features=32, bias=True)
  )
  (linear): Linear(in_features=32, out_features=2, bias=True)
)


In [6]:
pred

tensor([[[[0.1064, 1.9156],
          [0.1490, 2.1044],
          [0.1664, 2.1936],
          ...,
          [0.3324, 2.7490],
          [0.3350, 2.7555],
          [0.3373, 2.7619]],

         [[0.1130, 1.8395],
          [0.1516, 2.0411],
          [0.1692, 2.1398],
          ...,
          [0.2954, 2.7104],
          [0.2974, 2.7165],
          [0.2993, 2.7225]],

         [[0.1744, 1.4084],
          [0.2004, 1.6410],
          [0.2143, 1.7676],
          ...,
          [0.2797, 2.4858],
          [0.2805, 2.4934],
          [0.2809, 2.5009]],

         ...,

         [[0.3719, 3.4217],
          [0.3819, 3.4670],
          [0.3783, 3.5329],
          ...,
          [0.3542, 3.7206],
          [0.3632, 3.7201],
          [0.3718, 3.7197]],

         [[1.3958, 3.5865],
          [1.4167, 3.6960],
          [1.3839, 3.7769],
          ...,
          [1.3422, 3.8360],
          [1.3845, 3.8383],
          [1.4220, 3.8405]],

         [[0.9068, 2.6284],
          [0.8766, 2.6913],
    